# Paper 1 — Notebook 1b (v2): descarga complementaria de GFW (diario por buque + SAR)

Produce los datos de validación que usa la etapa N del Notebook 3. **No reemplaza** al Notebook 1 (v6): usa su `cells.csv`.

| descarga | para qué | salida |
|---|---|---|
| Esfuerzo de pesca **diario por buque** (2020–2025) | M2: presencia real por día, backtest fechado, buques distintos · M4: buques y banderas dentro de zonas no-take | `daily_block_vessel.parquet`, `vessels.csv`, `notake_daily_detail.csv` |
| Detecciones **SAR** mensuales, emparejadas y no emparejadas con AIS (2019–2025) | M1: ¿el esfuerzo AIS está donde están los buques que no transmiten? | `sar_monthly.parquet` |

**Cambio de la v2:** desde el 3 de noviembre de 2026 la API de GFW rechaza (error 404) los reportes agrupados por buque sobre una grilla. Esta versión usa la alternativa que GFW indica: un pedido por bloque de patrulla con agregación espacial. Son varios miles de pedidos chicos, por eso tarda varias horas. Además comprueba qué versión del producto de esfuerzo coincide con tu panel mensual.

**Uso:** Run all. Pide el token de GFW. La descarga es larga (la API permite un solo reporte a la vez), pero **es reanudable**:
si se corta, vuelve a correr todo y solo pide lo que falta. Al final se descarga `addon_small.zip` (archivos chicos) —
envíamelo. El archivo grande queda en Drive y lo lee directamente el Notebook 2.

In [ ]:
!pip -q install pyarrow requests shapely 2>/dev/null
import numpy as np, pandas as pd, json, os, time, glob, math, requests
from getpass import getpass
IN = os.environ.get('P1_IN', '/content/drive/MyDrive/acanav_paper1')
if 'P1_IN' not in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
OUT = f'{IN}/addon'
for d_ in ['raw_cellday','raw_vessel','raw_nt','raw_sar']: os.makedirs(f'{OUT}/{d_}', exist_ok=True)
cells = pd.read_csv(f'{IN}/cells.csv'); print(f'cells: {len(cells):,}')
GFW_TOKEN = os.environ.get('GFW_TOKEN') or getpass('Pega tu GFW API token (no se mostrara): ').strip()
print('token cargado:', bool(GFW_TOKEN))

In [ ]:
GFW_API     = 'https://gateway.api.globalfishingwatch.org/v3'
YEARS_DAILY = list(range(2021, 2026))     # daily effort (2021 feeds the presence estimates of the first test months)
TOP_SHARE   = float(os.environ.get('P1_TOP_SHARE', 0.98))   # vessel-level download: blocks that hold this share of the 2021-2025 effort
YEARS_SAR   = list(range(2019, 2026))
TILE_DEG    = 15.0                        # request tiles
BLK_DEG     = 0.6                         # must equal BLK_DEG of Notebook 2
BIN_EPS     = 1e-4
MAX_ATTEMPTS= 7

# ---------------- request tiles: only where there are cells, shrunk to the cells they contain ----------------
cells['tx']=np.floor(cells.lon/TILE_DEG).astype(int); cells['ty']=np.floor(cells.lat/TILE_DEG).astype(int)
TILES=[]
for (tx,ty),g in cells.groupby(['tx','ty']):
    d=float(g.delta.max())/2+1e-6                              # half a cell beyond the outermost cell centers: tiles do not overlap
    x0,x1,y0,y1=g.lon.min()-d,g.lon.max()+d,g.lat.min()-d,g.lat.max()+d
    TILES.append(dict(name=f't{tx}_{ty}',n=len(g),geo={'type':'Polygon','coordinates':[[[x0,y0],[x1,y0],[x1,y1],[x0,y1],[x0,y0]]]}))
print(len(TILES),'tiles:',[(t['name'],t['n']) for t in TILES])

# ---------------- API ----------------
def _post(url, geometry, timeout=900):
    return requests.post(url, headers={'Authorization':f'Bearer {GFW_TOKEN}','Content-Type':'application/json'}, data=json.dumps({'geojson':geometry}), timeout=timeout)

def _last_report(max_wait=900):
    """After a gateway timeout (524) the report keeps running; poll the last-report endpoint until it is ready."""
    t0=time.time()
    while time.time()-t0<max_wait:
        try:
            r=requests.get(f'{GFW_API}/4wings/last-report', headers={'Authorization':f'Bearer {GFW_TOKEN}'}, timeout=300)
            if r.status_code==200:
                js=r.json()
                if isinstance(js,dict) and str(js.get('status','')).lower() in ('running','pending','in_progress'): time.sleep(20); continue
                return js
        except requests.RequestException: pass
        time.sleep(20)
    return None

def fetch_report(geometry, dataset, d0, d1, temporal, group_by=None, filt=None, spatial_agg=False):
    url=(f'{GFW_API}/4wings/report?spatial-resolution=LOW&temporal-resolution={temporal}&format=JSON'
         f'&datasets[0]={dataset}&date-range={d0},{d1}')
    if group_by: url+=f'&group-by={group_by}'
    if spatial_agg: url+='&spatial-aggregation=true'
    if filt: url+=f'&filters[0]={requests.utils.quote(filt)}'
    last=''
    for a in range(MAX_ATTEMPTS):
        try:
            r=_post(url, geometry)
            if r.status_code==200: return r.json()
            last=f'{r.status_code}: {r.text[:200]}'
            if r.status_code==524:
                js=_last_report()
                if js is not None: return js
            elif r.status_code==429: time.sleep(30)                 # another report of this token is still running
            elif r.status_code not in (500,502,503,504): raise RuntimeError(last)        # 4xx: not retried
        except requests.RequestException as ex: last=f'network: {ex}'
        time.sleep(min(10*(2**a),300))
    raise RuntimeError(f'failed after {MAX_ATTEMPTS} attempts -- {last}')

KEYMAP={'lat':'lat','latitude':'lat','lon':'lon','longitude':'lon','long':'lon','date':'date','hours':'value','detections':'value',
        'vesselid':'vessel_id','vessel_id':'vessel_id','flag':'flag','geartype':'geartype','vesselids':'n_vessels'}
def flatten(js):
    """Collects every record that has latitude, longitude and a value, whatever the nesting of the response."""
    recs=[]
    def walk(n):
        if isinstance(n,dict):
            ks={k.lower() for k in n}
            if ('lat' in ks or 'latitude' in ks) and ('lon' in ks or 'longitude' in ks or 'long' in ks) and ('hours' in ks or 'detections' in ks): recs.append(n)
            else:
                for v in n.values(): walk(v)
        elif isinstance(n,list):
            for v in n: walk(v)
    walk(js)
    if not recs: return pd.DataFrame(columns=['lat','lon','date','value'])
    df=pd.DataFrame(recs); df=df.rename(columns={c:KEYMAP[c.lower()] for c in df.columns if c.lower() in KEYMAP})
    return df[[c for c in ['lat','lon','date','value','vessel_id','flag','geartype','n_vessels'] if c in df.columns]]

def bin_to_cells(e):
    """Assigns each GFW record to one cell (the finer cell where two resolutions overlap)."""
    e=e.reset_index(drop=True); e['rid']=np.arange(len(e)); parts=[]
    for d in sorted(cells.delta.unique()):
        cd=cells.loc[cells.delta==d,['cell_id','lon','lat']]; t=e.drop(columns=['lon','lat']).copy()
        t['lon']=(np.floor((e.lon+BIN_EPS)/d)*d+d/2).round(6); t['lat']=(np.floor((e.lat+BIN_EPS)/d)*d+d/2).round(6)
        parts.append(t.merge(cd,on=['lon','lat'],how='inner'))
    return pd.concat(parts,ignore_index=True).drop_duplicates('rid',keep='first').drop(columns=['rid','lon','lat'])
VKEYS=('vesselid','vessel_id','mmsi','ssvid')
def flatten_agg(js):
    """Records of a report with spatial aggregation: one per vessel and date, without coordinates."""
    recs=[]
    def walk(n):
        if isinstance(n,dict):
            ks={k.lower() for k in n}
            if 'hours' in ks and any(k in ks for k in VKEYS): recs.append({k.lower():v for k,v in n.items() if not isinstance(v,(dict,list))})
            else:
                for v in n.values(): walk(v)
        elif isinstance(n,list):
            for v in n: walk(v)
    walk(js)
    if not recs: return pd.DataFrame(columns=['date','value','vessel_id','flag','geartype'])
    df=pd.DataFrame(recs); vk=[k for k in VKEYS if k in df.columns][0]
    df=df.rename(columns={vk:'vessel_id','hours':'value'})
    for c_ in ['date','flag','geartype']:
        if c_ not in df.columns: df[c_]=''
    return df[['date','value','vessel_id','flag','geartype']]
from shapely.geometry import box, mapping
from shapely.ops import unary_union
def cells_geo(g):
    """Exact footprint of a set of cells (union of their squares) as GeoJSON."""
    u=unary_union([box(x-d/2,y-d/2,x+d/2,y+d/2) for x,y,d in zip(g.lon,g.lat,g.delta)]).buffer(0)
    return mapping(u)
try: mon=pd.read_parquet(f'{IN}/monthly_sparse.parquet')
except Exception: mon=pd.read_csv(f'{IN}/monthly_sparse.csv')
cells['blk_lon']=(np.floor(cells.lon/BLK_DEG)*BLK_DEG+BLK_DEG/2).round(4); cells['blk_lat']=(np.floor(cells.lat/BLK_DEG)*BLK_DEG+BLK_DEG/2).round(4)
print('helpers listos')

## 0. ¿Qué versión del producto de esfuerzo coincide con tu panel mensual?
GFW cambió el modelo de detección de pesca el 21 de octubre de 2026 (`latest` pasó a la versión 5, con menos horas). El paper usa la versión 4 (`v4.0`). Esta celda compara un mes de un mosaico con el panel mensual en ambas versiones y elige la que coincide.

In [ ]:
CAND=['public-global-fishing-effort:v4.0','public-global-fishing-effort:latest']   # the paper uses v4.0; latest (v5) is probed for comparison only
PROBE_MONTHS=[('2023-03-01','2023-03-31'),('2024-09-01','2024-09-30')]
tb=max(TILES,key=lambda t:t['n']); probe=[]
for ds in CAND:
    for d0,d1 in PROBE_MONTHS:
        try:
            df=flatten(fetch_report(tb['geo'],ds,d0,d1,'MONTHLY'))
            e_=bin_to_cells(df) if len(df) else df; api=float(e_.value.sum()) if len(e_) else 0.0
            ids=cells[(cells.tx==int(tb['name'][1:].split('_')[0]))&(cells.ty==int(tb['name'].split('_')[1]))].cell_id
            y_,m_=int(d0[:4]),int(d0[5:7]); pan=float(mon[(mon.year==y_)&(mon.month==m_)&mon.cell_id.isin(ids)].all_hours.sum())
            probe.append(dict(dataset=ds,month=d0[:7],api_hours=api,panel_hours=pan,ratio=api/pan if pan>0 else np.nan))
        except Exception as ex: probe.append(dict(dataset=ds,month=d0[:7],api_hours=np.nan,panel_hours=np.nan,ratio=np.nan,error=str(ex)[:140]))
probe=pd.DataFrame(probe); print(probe.round(3).to_string(index=False)); probe.to_csv(f'{OUT}/version_probe.csv',index=False)
score=probe.dropna(subset=['ratio']).groupby('dataset').ratio.apply(lambda r:float(np.abs(np.log(r)).mean()))
assert len(score), 'Ninguna versión respondió: revisa el token y mándame la tabla de arriba.'
DATASET=[d_ for d_ in CAND if d_ in score.index and score[d_]<=score.min()+0.005][0]; MATCH=bool(score.min()<0.03)
print(f'\n>>> versión elegida: {DATASET}  (desviación media {100*score.min():.1f} %)')
if not MATCH: print('>>> ATENCION: ninguna versión reproduce el panel mensual dentro de 3 %. La descarga sigue con la más cercana, pero MÁNDAME ESTA TABLA antes de correr el Notebook 2.')

## 1. Esfuerzo diario por celda (sin identidad del buque)
Rápido: un pedido por mosaico y trimestre. Da las horas de cada día en cada bloque y dentro de las zonas no-take.

In [ ]:
def quarters(y): return [(f'{y}-01-01',f'{y}-03-31'),(f'{y}-04-01',f'{y}-06-30'),(f'{y}-07-01',f'{y}-09-30'),(f'{y}-10-01',f'{y}-12-31')]
def months_of(d0,d1): return [(str(p.start_time.date()),str(p.end_time.date())) for p in pd.period_range(d0,d1,freq='M')]
TAG=DATASET.split(':')[-1]
def get_cellday(tile,d0,d1):
    fn=f'{OUT}/raw_cellday/{TAG}_{tile["name"]}_{d0}_{d1}.parquet'
    if os.path.exists(fn): return pd.read_parquet(fn)
    df=flatten(fetch_report(tile['geo'],DATASET,d0,d1,'DAILY'))
    if len(df):
        if 'n_vessels' not in df.columns: df['n_vessels']=np.nan
        df=df[['lat','lon','date','value','n_vessels']]
    df.to_parquet(fn,index=False); return df
jobs=[(t,d0,d1) for t in TILES for y in YEARS_DAILY for d0,d1 in quarters(y)]
t0=time.time(); err1=[]; nrows=0
for k,(t,d0,d1) in enumerate(jobs,1):
    try: nrows+=len(get_cellday(t,d0,d1))
    except Exception as ex:
        try:
            for m0,m1 in months_of(d0,d1): nrows+=len(get_cellday(t,m0,m1))
        except Exception as ex2: err1.append(f'{t["name"]} {d0}: {str(ex2)[:200]}')
    if k%20==0 or k==len(jobs): print(f'  [{k}/{len(jobs)}] filas={nrows:,}  errores={len(err1)}  ETA {(time.time()-t0)/k*(len(jobs)-k)/60:.0f} min',flush=True)
print(f'diario por celda: {(time.time()-t0)/60:.1f} min, errores: {len(err1)}'); [print('  ',e) for e in err1[:5]]
fr=[pd.read_parquet(f) for f in sorted(glob.glob(f'{OUT}/raw_cellday/{TAG}_*.parquet'))]; fr=[f for f in fr if len(f)]
assert fr, 'La descarga diaria por celda no devolvió datos: mándame los errores de arriba.'
raw=pd.concat(fr,ignore_index=True); raw['date']=pd.to_datetime(raw.date,errors='coerce').dt.tz_localize(None).dt.normalize(); raw=raw.dropna(subset=['date']).drop_duplicates(subset=['lat','lon','date'])
e=bin_to_cells(raw).merge(cells[['cell_id','zone','no_take','mpa_name','blk_lon','blk_lat']],on='cell_id')
e['hours_nt']=np.where((e.zone=='interior')&(e.no_take==1),e.value,0.0)
dbd=e.groupby(['blk_lon','blk_lat','date'],as_index=False).agg(hours=('value','sum'),hours_nt=('hours_nt','sum'),n_vessels_max=('n_vessels','max'))
dbd.to_parquet(f'{OUT}/daily_block.parquet',index=False)
ntc=e[e.hours_nt>0][['cell_id','mpa_name','date','value','n_vessels']].rename(columns={'value':'hours'}); ntc.to_csv(f'{OUT}/notake_cell_day.csv',index=False)
chk=pd.DataFrame({'daily':e.groupby(e.date.dt.year).value.sum(),'monthly':mon[mon.year.isin(YEARS_DAILY)].groupby('year').all_hours.sum()}); chk['ratio']=chk.daily/chk.monthly
print(f'bloque-dia: {len(dbd):,} filas | filas no-take: {len(ntc):,}'); print(chk.round(3).to_string()); chk.to_csv(f'{OUT}/check_daily_vs_monthly.csv')

## 2. Esfuerzo diario por buque, bloque por bloque
Un pedido por bloque y año, con agregación espacial (la única forma en que la API entrega hoy la identidad del buque). Solo se piden los bloques que concentran el 98 % del esfuerzo y los años en que el panel registra esfuerzo en el bloque. **Tarda varias horas y es reanudable.**

In [ ]:
mb=mon[mon.year.isin(YEARS_DAILY)].merge(cells[['cell_id','blk_lon','blk_lat']],on='cell_id').groupby(['blk_lon','blk_lat','year'],as_index=False).all_hours.sum()
tot=mb.groupby(['blk_lon','blk_lat']).all_hours.sum().sort_values(ascending=False); keep=tot.index[:int((tot.cumsum()/tot.sum()<TOP_SHARE).sum())+1]
BLK={k:cells_geo(g) for k,g in cells.groupby(['blk_lon','blk_lat']) if k in set(keep)}
vjobs=[(bl,ba,int(y)) for (bl,ba,y,h) in mb.itertuples(index=False) if (bl,ba) in BLK and h>0]
print(f'bloques: {len(BLK):,} de {len(tot):,} con esfuerzo ({100*tot[keep].sum()/tot.sum():.1f} % de las horas) | pedidos: {len(vjobs):,}')
def get_vessel(bl,ba,y):
    fn=f'{OUT}/raw_vessel/{TAG}_{bl:.1f}_{ba:.1f}_{y}.parquet'
    if os.path.exists(fn): return pd.read_parquet(fn)
    df=flatten_agg(fetch_report(BLK[(bl,ba)],DATASET,f'{y}-01-01',f'{y}-12-31','DAILY',group_by='VESSEL_ID',spatial_agg=True))
    df['blk_lon']=bl; df['blk_lat']=ba; df.to_parquet(fn,index=False); return df
# one request first, shown in full, so that a change in the format of the response is seen at once
bl0,ba0,y0=max(vjobs,key=lambda j: float(mb[(mb.blk_lon==j[0])&(mb.blk_lat==j[1])&(mb.year==j[2])].all_hours.iloc[0]))
try:
    fn0=f'{OUT}/raw_vessel/{TAG}_{bl0:.1f}_{ba0:.1f}_{y0}.parquet'
    if os.path.exists(fn0): d0_=pd.read_parquet(fn0)
    else:
        js0=fetch_report(BLK[(bl0,ba0)],DATASET,f'{y0}-01-01',f'{y0}-12-31','DAILY',group_by='VESSEL_ID',spatial_agg=True); d0_=flatten_agg(js0)
        print('respuesta de prueba:',json.dumps(js0)[:700])
        if len(d0_): d0_['blk_lon']=bl0; d0_['blk_lat']=ba0; d0_.to_parquet(fn0,index=False)
    print(d0_.head(3).to_string()); print('filas:',len(d0_),'| fechas distintas:',d0_.date.nunique() if len(d0_) else 0)
    VESSEL_OK=len(d0_)>0 and d0_.date.astype(str).str.len().max()>=10
    if not VESSEL_OK: print('>>> ATENCION: la respuesta no trae filas diarias por buque. Copia y mándame el texto de "respuesta de prueba".')
except Exception as ex: VESSEL_OK=False; print('>>> ATENCION: el pedido por buque falló:',str(ex)[:400])
t0=time.time(); err2=[]; nrows=0; MAX_ERR=40
if VESSEL_OK:
    todo=[j for j in vjobs if not os.path.exists(f'{OUT}/raw_vessel/{TAG}_{j[0]:.1f}_{j[1]:.1f}_{j[2]}.parquet')]; print(f'pendientes: {len(todo):,} de {len(vjobs):,}')
    for k,(bl,ba,y) in enumerate(todo,1):
        try: nrows+=len(get_vessel(bl,ba,y))
        except Exception as ex:
            err2.append(f'{bl:.1f},{ba:.1f} {y}: {str(ex)[:200]}')
            if len(err2)>=MAX_ERR and len(err2)>0.5*k: print('demasiados errores seguidos: se detiene la descarga'); break
        if k%50==0 or k==len(todo): print(f'  [{k}/{len(todo)}] filas={nrows:,}  errores={len(err2)}  ETA {(time.time()-t0)/k*(len(todo)-k)/60:.0f} min',flush=True)
    print(f'diario por buque: {(time.time()-t0)/60:.1f} min, errores: {len(err2)}'); [print('  ',e_) for e_ in err2[:5]]
    if err2: print('>>> Hay pedidos fallidos: vuelve a correr esta celda (solo se piden los que faltan).')

In [ ]:
# ---------------- assemble: block x day x vessel, and its coverage ----------------
fr=[]; cov=[]
for (bl,ba,y) in vjobs:
    fn=f'{OUT}/raw_vessel/{TAG}_{bl:.1f}_{ba:.1f}_{y}.parquet'
    if os.path.exists(fn):
        cov.append((bl,ba,y)); d=pd.read_parquet(fn)
        if len(d): fr.append(d)
# block-years without effort in the panel were not requested: they are covered by construction
cov+= [(bl,ba,int(y)) for (bl,ba,y,h) in mb.itertuples(index=False) if (bl,ba) in BLK and h<=0]
cov+= [(bl,ba,y) for (bl,ba) in BLK for y in YEARS_DAILY if (bl,ba,y) not in set(map(tuple,mb[['blk_lon','blk_lat','year']].values))]
pd.DataFrame(sorted(set(cov)),columns=['blk_lon','blk_lat','year']).to_csv(f'{OUT}/daily_coverage.csv',index=False)
HAVE_V=len(fr)>0
if HAVE_V:
    v=pd.concat(fr,ignore_index=True); v['date']=pd.to_datetime(v.date,errors='coerce').dt.tz_localize(None).dt.normalize(); v=v.dropna(subset=['date'])
    v['value']=pd.to_numeric(v.value,errors='coerce').fillna(0.0); v=v[v.value>0]
    codes,uniq=pd.factorize(v.vessel_id); v['vessel']=codes.astype(np.int32)
    ves=v.groupby('vessel').agg(flag=('flag','first'),geartype=('geartype','first'),hours=('value','sum'),days=('date','nunique'),blocks=('blk_lon','size')).reset_index(); ves.to_csv(f'{OUT}/vessels.csv',index=False)
    dbv=v.groupby(['blk_lon','blk_lat','date','vessel'],as_index=False).agg(hours=('value','sum')); dbv['hours']=dbv.hours.astype(np.float32); dbv['hours_nt']=np.float32(0.0)
    dbv.to_parquet(f'{OUT}/daily_block_vessel.parquet',index=False)
    c2=dbv.groupby([dbv.blk_lon,dbv.blk_lat,dbv.date.dt.year.rename('year')]).hours.sum().rename('vessel_h').reset_index().merge(mb,on=['blk_lon','blk_lat','year'],how='left')
    chk2=c2.groupby('year')[['vessel_h','all_hours']].sum(); chk2['ratio']=chk2.vessel_h/chk2.all_hours
    print(f'bloque-dia-buque: {len(dbv):,} filas | buques: {len(ves):,} | bloque-años cubiertos: {len(set(cov)):,}'); print('horas por buque vs panel, en los bloque-años descargados:'); print(chk2.round(3).to_string()); chk2.to_csv(f'{OUT}/check_vessel_vs_monthly.csv')
else: print('SIN DATOS POR BUQUE: el Notebook 2 usará solo el diario por bloque.')

## 3. Buques dentro de las zonas no-take
Un pedido por sitio no-take y año sobre la huella exacta de sus celdas interiores.

In [ ]:
ntcells=cells[(cells.zone=='interior')&(cells.no_take==1)]; nt_rows=[]; err3=[]
if VESSEL_OK:
    for name,g in ntcells.groupby('mpa_name'):
        geo=cells_geo(g); tagn=''.join(ch for ch in name if ch.isalnum())[:30]
        for y in YEARS_DAILY:
            fn=f'{OUT}/raw_nt/{TAG}_{tagn}_{y}.parquet'
            try:
                if os.path.exists(fn): d=pd.read_parquet(fn)
                else:
                    d=flatten_agg(fetch_report(geo,DATASET,f'{y}-01-01',f'{y}-12-31','DAILY',group_by='VESSEL_ID',spatial_agg=True)); d.to_parquet(fn,index=False)
                if len(d): d=d.copy(); d['mpa_name']=name; nt_rows.append(d)
            except Exception as ex: err3.append(f'{name} {y}: {str(ex)[:160]}')
    print('errores no-take:',len(err3)); [print('  ',e_) for e_ in err3[:5]]
if nt_rows:
    nt=pd.concat(nt_rows,ignore_index=True).rename(columns={'value':'hours','vessel_id':'vessel'}); nt['hours']=pd.to_numeric(nt.hours,errors='coerce'); nt=nt[nt.hours>0]
    nt[['mpa_name','date','vessel','flag','geartype','hours']].to_csv(f'{OUT}/notake_daily_detail.csv',index=False)
    print(nt.groupby(['mpa_name','flag','geartype']).agg(hours=('hours','sum'),vessels=('vessel','nunique'),days=('date','nunique')).round(1).sort_values('hours',ascending=False).head(20).to_string())
else: print('sin detalle por buque en zonas no-take (queda el detalle por celda y día)')

## 2. Detecciones SAR mensuales (emparejadas y no emparejadas con AIS)

In [ ]:
def get_sar(tile,y,matched):
    fn=f'{OUT}/raw_sar/{tile["name"]}_{y}_{matched}.parquet'
    if os.path.exists(fn): return pd.read_parquet(fn)
    last=None
    for gb in (None,'FLAG','VESSEL_ID'):                 # the endpoint may or may not require a group-by for this dataset
        try:
            df=flatten(fetch_report(tile['geo'],'public-global-sar-presence:latest',f'{y}-01-01',f'{y}-12-31','MONTHLY',group_by=gb,filt=f"matched='{matched}'")); break
        except Exception as ex: last=ex; df=None
    if df is None: raise last
    df=df[['lat','lon','date','value']] if len(df) else df; df.to_parquet(fn,index=False); return df
t0=time.time(); sar=[]; sar_err=[]
for k,(t,y,mt) in enumerate([(t,y,mt) for t in TILES for y in YEARS_SAR for mt in ('false','true')],1):
    try:
        d=get_sar(t,y,mt)
        if len(d): d=d.copy(); d['matched']=mt; sar.append(d)
    except Exception as ex: sar_err.append(f'{t["name"]} {y} {mt}: {str(ex)[:160]}')
    if k%20==0: print(f'  SAR [{k}]  errores={len(sar_err)}  {(time.time()-t0)/60:.1f} min',flush=True)
print('errores SAR:',len(sar_err)); [print('  ',e_) for e_ in sar_err[:5]]
if sar:
    s=pd.concat(sar,ignore_index=True); s['date']=pd.to_datetime(s.date,errors='coerce').dt.tz_localize(None); s=s.dropna(subset=['date'])
    s=s.drop_duplicates(subset=['lat','lon','date','matched']); sb=bin_to_cells(s)
    sb['year']=sb.date.dt.year; sb['month']=sb.date.dt.month
    sm=sb.pivot_table(index=['cell_id','year','month'],columns='matched',values='value',aggfunc='sum',fill_value=0).reset_index().rename(columns={'false':'det_unmatched','true':'det_matched'})
    for c in ['det_unmatched','det_matched']:
        if c not in sm.columns: sm[c]=0.0
    sm.to_parquet(f'{OUT}/sar_monthly.parquet',index=False)
    print(f'SAR celda-mes: {len(sm):,} | no emparejadas: {sm.det_unmatched.sum():,.0f} | emparejadas: {sm.det_matched.sum():,.0f}')
    print(sm.merge(cells[['cell_id','region']],on='cell_id').groupby('region')[['det_unmatched','det_matched']].sum().round(0).to_string())
else: print('SIN DATOS SAR (revisar errores)')

In [ ]:
# ---------------- summary and small-files zip ----------------
import zipfile
L=[f'dataset: {DATASET} | matches the monthly panel within 3%: {MATCH}','version probe:',probe.round(3).to_string(index=False),f'block-day rows: {len(dbd):,}','daily (cells) vs monthly:',chk.round(3).to_string(),
   f'vessel-level data: {HAVE_V}',f'errors: cell-day {len(err1)}, vessel {len(err2)}, no-take {len(err3)}, SAR {len(sar_err)}']
if HAVE_V: L+=[f'block-day-vessel rows: {len(dbv):,} | vessels: {len(ves):,} | blocks requested: {len(BLK):,} | block-years covered: {len(set(cov)):,} of {len(BLK)*len(YEARS_DAILY):,}','vessel hours vs monthly panel:',chk2.round(3).to_string(),
              'flags (hours):',ves.groupby('flag').hours.sum().sort_values(ascending=False).head(12).round(0).to_string()]
open(f'{OUT}/SUMMARY_addon.txt','w').write('\n'.join(L)); print('\n'.join(L))
z=f'{OUT}/addon_small.zip'
with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in ['SUMMARY_addon.txt','version_probe.csv','vessels.csv','notake_daily_detail.csv','notake_cell_day.csv','check_daily_vs_monthly.csv','check_vessel_vs_monthly.csv','daily_coverage.csv','sar_monthly.parquet','daily_block.parquet']:
        if os.path.exists(f'{OUT}/{f}'): zf.write(f'{OUT}/{f}',f)
    if HAVE_V and os.path.getsize(f'{OUT}/daily_block_vessel.parquet')<25e6: zf.write(f'{OUT}/daily_block_vessel.parquet','daily_block_vessel.parquet')
try:
    from google.colab import files; files.download(z)
except Exception as ex: print('descarga manual:',z)